### TODO
- DPO
- HVM / CMD
- Distance aware
    - warm start general sinkhorn
- gh6 mapping / 2D

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from core import Graph, GraphPricing, DPO, ConversionLearner

In [ ]:
np.set_printoptions(precision=3, suppress=True)

# Optimal Transport

In [ ]:
N = 10
sink = np.zeros(N)
source = np.zeros(N)
source[1] = 8
source[5] = 2
sink[4] = 2
sink[9] = 8
gammas = np.ones(N) * 0.01
gammas[1] = 0.008
graph = Graph(source, sink, gammas=gammas)

In [ ]:
transport, history = graph.sinkhorn(max_iter=4)
print(history)

In [ ]:
transport, history = graph.general_sinkhorn(max_iter=4)
print(history)

In [ ]:
N = 100
v = 10
source = np.zeros(N)
for i in range(N):
    source[i] += 2 * math.exp(-(i - N*0.40)**2 / v**2)
    source[i] += math.exp(-(i - N*0.70)**2 / v**2)
sink = np.zeros(N)
for i in range(N):
    sink[i] = 0.5 * math.exp(-(i - N*0.45)**2 / (3*v)**2)
source /= source.sum()
sink /= sink.sum()
gammas = np.full(N, 0.01)
gammas[40:45] = 0.008
graph_gaussians = Graph(source, sink, gammas=gammas)

In [ ]:
transport, history = graph_gaussians.sinkhorn()
print(len(history), history[-1])

In [ ]:
transport, history = graph_gaussians.general_sinkhorn()
print(len(history), history[-1])

In [ ]:
# plt.plot(history)
# plt.show()

In [ ]:
plt.matshow(graph_gaussians.transport)
plt.show()

# Graph Pricing

In [ ]:
N = 100
demand = np.ones(N) * 0.5
demand[40:60] = 10
supply = np.ones(N)
hvm_ratio = 0.9
hvm_supply = supply * (1 - hvm_ratio)
graph = GraphPricing(demand, supply, hvm_supply=hvm_supply, hvm_ratio=hvm_ratio)

In [ ]:
N = 100
v = 10
demand = np.zeros(N)
for i in range(N):
    demand[i] += 2 * math.exp(-(i - N*0.40)**2 / v**2)
    demand[i] += math.exp(-(i - N*0.70)**2 / v**2)
supply = np.zeros(N)
for i in range(N):
    supply[i] = 0.5 * math.exp(-(i - N*0.45)**2 / (3*v)**2)
demand /= demand.sum() / N
supply /= supply.sum() / N
demand += 0.1
demand *= 4
supply += 1

hvm_ratio = 0.9
hvm_supply = supply * (1 - hvm_ratio)
graph = GraphPricing(demand, supply, hvm_supply=hvm_supply, hvm_ratio=hvm_ratio)

In [ ]:
# plt.matshow(graph.edge_costs["flex_d1", "standard"])
# plt.show()

In [ ]:
# transport, history = graph.sinkhorn()
transport, history = graph.general_sinkhorn(max_iter=1000)
print(len(history))
verified = graph.verify_constraints()
print(verified)
print(graph.objective())
print(graph.total_cost())

In [ ]:
plt.plot(history[1:])
plt.show()

In [ ]:
transport = graph.transport
transport = np.clip(transport, 0, 0.05)
plt.matshow(transport)
plt.show()

In [ ]:
transports = graph.split_transport()
plt.matshow(transports[("flex_d2", "standard")])
# plt.matshow(transports[("hvm_flex", "hvm")])
plt.show()

In [ ]:
demand = graph.demand_nodes
requests = graph.split_requests()
converted = graph.converted_requests()
cvr = graph.conversion_target()
pt = graph.primetime()

# plt.plot(graph.session)
# plt.plot(graph.total_demand())
# plt.plot(graph.min_demand())
# plt.plot(graph.supply)

plt.plot(demand["min"] + demand["flex_d1"], linestyle=":")
plt.plot(demand["min"] + demand["flex_d2"], linestyle=":")
plt.plot(demand["min"] + demand["flex_d1"] + demand["flex_d2"], linestyle=":")
plt.plot(demand["min"], linestyle=":")
plt.plot(requests[("min", "standard")] + requests[("min", "hvm")])
plt.plot(requests[("min", "standard")] + requests[("flex_d1", "standard")])
plt.plot(requests[("min", "standard")] + requests[("flex_d2", "standard")])
plt.plot(converted["min"] + converted["flex_d1"] + converted["flex_d2"])
plt.show()

In [ ]:
plt.plot(demand["hvm_min"] + demand["hvm_flex"], linestyle=":")
plt.plot(demand["hvm_min"], linestyle=":")
plt.plot(requests[("hvm_min", "hvm")])
plt.plot(converted["hvm"])
plt.show()

In [ ]:
fig, ax1 = plt.subplots()
ax1.plot(pt["standard"])
ax1.plot(pt["d1"])
ax1.plot(pt["d2"])
ax1.plot(pt["hvm"])
ax2 = ax1.twinx()
linestyle = ":"
ax2.plot(cvr["standard"], linestyle=linestyle)
ax2.plot(cvr["d1"], linestyle=linestyle)
ax2.plot(cvr["d2"], linestyle=linestyle)
ax2.plot(cvr["hvm"], linestyle=linestyle)
ax2.set_ylim(0, 1)
plt.show()

# DPO

In [ ]:
price = np.arange(0, 10, 0.1)
dpo = DPO(etas=[3], mode_mix=[1], base_prices=[1])
cvrs = [dpo.conversion([p])[0] for p in price]
plt.plot(price, cvrs)
booking = [dpo.booking([p]) for p in price]
plt.plot(price, booking)
plt.show()

In [ ]:
ct = 0.8
dpo = DPO(etas=[0, 5.7], mode_mix=[0.1, 0.6], base_prices=[3.3, 2.7])
price_st = np.arange(0, 10, 0.1)
price_ws = np.arange(0, 10, 0.1)
res = np.zeros((len(price_st), len(price_ws)))
for i, p_st in enumerate(price_st):
    for j, p_ws in enumerate(price_ws):
        prices = [p_st, p_ws]
        # res[i,j] = sum(dpo.conversion(prices))
        # res[i,j] = dpo.conversion(prices)[0]
        # res[i,j] = dpo.booking(prices)
        res[i,j] = dpo.objective(prices, ct)
        # res[i,j] = -1000 * dpo.constraint(prices, ct) ** 2
res = np.clip(res, min=-1)
plt.matshow(res)
plt.show()

In [ ]:
# plt.plot(res[99,:])
# plt.plot(res[:,99])
# plt.show()

In [ ]:
dpo = DPO()
prices = dpo.base_prices
ct = dpo.base_conversion
grad = dpo.gradient(prices, ct)
num_grad = dpo.numerical_gradient(prices, ct)
assert np.allclose(grad, num_grad)

In [ ]:
prices, history = dpo.gradient_descent(ct)
error = dpo.constraint(prices, ct)
print(len(history))
print(error)
print(prices)
print(dpo.base_prices)
print(dpo.conversion(prices))
print(dpo.base_conversion * dpo.mode_mix)
print(dpo.thetas)
print(dpo.theta0)

In [ ]:
plt.plot(history)
plt.show()

In [ ]:
cvrs = dpo.conversion(dpo.base_prices)
assert np.allclose(cvrs / sum(cvrs), dpo.mode_mix)
assert np.allclose(np.sum(dpo.conversion(dpo.base_prices)), dpo.base_conversion)

In [ ]:
cvrs = np.arange(0, 1, 0.01)
dpo = DPO()
prices = dpo.solve(cvrs)
bookings = [dpo.booking(p) for p in prices]
cvrs_mode = [dpo.conversion(p) for p in prices]

print(dpo.thetas)
print(dpo.theta0)
plt.plot(cvrs, prices)
plt.plot(cvrs, bookings)
plt.show()

In [ ]:
plt.plot(cvrs, cvrs_mode)
# plt.plot(cvrs, np.sum(cvrs_mode, axis=1))
plt.show()

In [ ]:
dpo = DPO()
cvr = graph.conversion_target()
prices = dpo.solve(cvr["standard"])

fig, ax1 = plt.subplots()
ax1.plot(prices)
ax2 = ax1.twinx()
ax2.plot(cvr["standard"], linestyle="--")
ax2.set_ylim(0, 1)
plt.show()